# Plots on the native cubed-sphere mesh

In [ ]:
import math
import warnings
from functools import partial

import cartopy.crs as ccrs
import iris
import matplotlib.colors as mcol
import matplotlib.pyplot as plt
import numpy as np
import paths
from aeolus.calc import (
    integrate,
    last_n_day_mean,
    spatial,
    spatial_mean,
    time_mean,
    toa_net_energy,
    zonal_mean,
)
from aeolus.const import init_const
from aeolus.coord import (
    get_cube_rel_days,
    interp_cube_from_height_to_pressure_levels,
    isel,
    roll_cube_pm180,
)
from aeolus.lfric import fix_time_coord, load_lfric_raw, ugrid_spatial
from aeolus.model import lfric
from aeolus.plot import (
    figsave,
    map_scatter,
    timeseries_1d,
    timeseries_2d,
    hcross,
    tex2cf_units,
    unit_format,
    subplot_label_generator,
)
from common import (
    EXPERIMENTS,
    GROUPS,
    PA_TO_BAR,
    Diag,
    align_cbar_with_axes,
    common_kw_plt,
    interp_to_plev,
    lfric_callback_uniform_height,
    load_exp_days,
)

warnings.filterwarnings("ignore")
iris.FUTURE.date_microseconds = True

CART_TRANS = dict(transform=ccrs.PlateCarree())

print(*EXPERIMENTS.keys(), sep="\n")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.collections import LineCollection, PolyCollection


def _unwrap_lons(lons, ref):
    """Shift longitudes to be within 180 degrees of the reference longitude."""
    return ref + (lons - ref + 180) % 360 - 180


def get_face_nodes(obj, decimals=6):
    """
    Get node coordinates and face-node connectivity of a cubed-sphere mesh.

    Parameters
    ----------
    obj: iris.mesh.MeshXY or iris.cube.Cube
        Either a mesh, a cube located on mesh faces (UGRID output, e.g.
        lfric_initial.nc), or a cube with bounded latitude and longitude
        coordinates of the cells (XIOS non-UGRID output, e.g. lfric_diag_main*).
    decimals: int, optional
        Precision used to merge cell corners into nodes (in units of the
        sphere radius), if the connectivity is built from coordinate bounds.

    Returns
    -------
    node_lons, node_lats: (n_nodes,) arrays
    face_nodes: (n_faces, n_corners) array of node indices
    """
    mesh = getattr(obj, "mesh", obj)
    if mesh is not None and hasattr(mesh, "face_node_connectivity"):
        if getattr(obj, "location", "face") != "face":
            raise ValueError("Cube must be located on mesh faces.")
        face_nodes = mesh.face_node_connectivity.indices_by_location()
        if np.ma.is_masked(face_nodes):
            raise NotImplementedError("Faces with different numbers of nodes.")
        return (
            mesh.node_coords.node_x.points,
            mesh.node_coords.node_y.points,
            np.asarray(face_nodes),
        )
    # No mesh: use the cell corners stored as lat-lon coordinate bounds
    try:
        lon_coord = obj.coord("longitude")
        lat_coord = obj.coord("latitude")
    except (AttributeError, iris.exceptions.CoordinateNotFoundError):
        raise ValueError(
            "Need a mesh or a cube with latitude and longitude coordinates."
        )
    if not (lon_coord.has_bounds() and lat_coord.has_bounds()):
        raise ValueError("Latitude and longitude coordinates must have bounds.")
    corner_lons = lon_coord.bounds
    corner_lats = lat_coord.bounds
    # Merge identical corners into nodes, comparing them as points on the unit
    # sphere to avoid ambiguity of longitudes (wrap-around and poles)
    lam, phi = np.deg2rad(corner_lons.ravel()), np.deg2rad(corner_lats.ravel())
    xyz = np.column_stack([
        np.cos(phi) * np.cos(lam),
        np.cos(phi) * np.sin(lam),
        np.sin(phi),
    ])
    _, first, inverse = np.unique(
        np.round(xyz, decimals) + 0.0,  # + 0.0 turns -0.0 into 0.0
        axis=0,
        return_index=True,
        return_inverse=True,
    )
    face_nodes = inverse.reshape(corner_lons.shape)
    return corner_lons.ravel()[first], corner_lats.ravel()[first], face_nodes


def mesh_face_polygons(node_lons, node_lats, face_nodes, pole_tol=1e-6):
    """
    Get vertices of mesh faces as lon-lat polygons that do not wrap around.

    Polygons are unwrapped in longitude relative to their first vertex,
    and polygons touching a pole get the pole vertex split in two,
    so that they look right in a lon-lat (PlateCarree) space.

    Returns
    -------
    verts: list of (n_vertices, 2) arrays of (lon, lat)
    """
    lons = node_lons[face_nodes]
    lats = node_lats[face_nodes]
    at_pole = np.abs(np.abs(lats) - 90) < pole_tol
    # Unwrap relative to the first non-pole vertex of each face
    ref_idx = np.argmax(~at_pole, axis=1)
    ref = np.take_along_axis(lons, ref_idx[:, None], axis=1)
    lons = _unwrap_lons(lons, ref)

    verts = []
    n = lons.shape[1]
    for i in range(lons.shape[0]):
        if not at_pole[i].any():
            verts.append(np.column_stack([lons[i], lats[i]]))
            continue
        # Replace the pole vertex with two vertices at the longitudes of its neighbours
        poly = []
        for j in range(n):
            if at_pole[i, j]:
                poly.append((lons[i, (j - 1) % n], lats[i, j]))
                poly.append((lons[i, (j + 1) % n], lats[i, j]))
            else:
                poly.append((lons[i, j], lats[i, j]))
        verts.append(np.array(poly))
    return verts


def plot_mesh_polygons(cube, ax=None, transform="auto", edgecolor="none", **kwargs):
    """
    Plot cubed-sphere mesh data on faces as polygons in lon-lat coordinates.

    Parameters
    ----------
    cube: iris.cube.Cube
        1D cube of cell values (slice other dimensions beforehand), either
        on mesh faces or with bounded latitude and longitude coordinates.
    ax: matplotlib.axes.Axes, optional
        Axes to plot on (e.g. cartopy GeoAxes). Defaults to the current axes.
    transform: cartopy.crs.CRS, "auto" or None, optional
        Coordinate system of the lon-lat polygons.
        - "auto" (default): `ccrs.PlateCarree()` if `ax` is a cartopy GeoAxes,
          otherwise None.
        - None: plot on regular (non-cartopy) matplotlib axes, with longitude
          and latitude in degrees as x and y.
        - a cartopy CRS.
    edgecolor: str, optional
        Polygon edge colour.
    **kwargs:
        Passed to `matplotlib.collections.PolyCollection` (cmap, norm, etc).

    Returns
    -------
    matplotlib.collections.PolyCollection
    """
    if ax is None:
        ax = plt.gca()
    if isinstance(transform, str) and transform == "auto":
        transform = ccrs.PlateCarree() if hasattr(ax, "projection") else None
    if cube.ndim != 1:
        raise ValueError(f"Cube must be 1D, got shape {cube.shape}.")
    verts = mesh_face_polygons(*get_face_nodes(cube))
    data = cube.data
    if len(verts) != data.shape[0]:
        raise ValueError(
            f"{len(verts)} faces do not match {data.shape[0]} data points."
        )
    if transform is None:
        # Plain axes: duplicate polygons crossing the dateline, shifted by 360 degrees
        # (cartopy wraps them automatically, so this is not needed for GeoAxes)
        extra_verts, extra_idx = [], []
        for i, v in enumerate(verts):
            if v[:, 0].max() > 180:
                extra_verts.append(v - [360, 0])
                extra_idx.append(i)
            elif v[:, 0].min() < -180:
                extra_verts.append(v + [360, 0])
                extra_idx.append(i)
        verts = verts + extra_verts
        data = np.ma.concatenate([data, data[extra_idx]])
    else:
        kwargs["transform"] = transform
    coll = PolyCollection(verts, array=data, edgecolor=edgecolor, **kwargs)
    ax.add_collection(coll)
    if transform is None:
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
    return coll


def cube_panel_edges(obj):
    """
    Find the 12 edges of the six panels of a cubed-sphere mesh.

    The panel edges are traced along the mesh topology: they start at the
    8 cube corners (nodes shared by 3 faces instead of 4) and continue
    "straight" through each node to the next corner.

    Parameters
    ----------
    obj: iris.mesh.MeshXY or iris.cube.Cube
        Mesh or cube, see `get_face_nodes()`.

    Returns
    -------
    list of (n_points, 2) arrays of (lon, lat), unwrapped in longitude.
    """
    node_lons, node_lats, face_nodes = get_face_nodes(obj)
    n_corners = face_nodes.shape[1]

    # Map each (sorted) node pair to the faces sharing it, and each node to its neighbours
    edge_faces = {}
    for i_face, nodes in enumerate(face_nodes):
        for j in range(n_corners):
            key = tuple(sorted((nodes[j], nodes[(j + 1) % n_corners])))
            edge_faces.setdefault(key, set()).add(i_face)
    neighbours = {}
    for a, b in edge_faces:
        neighbours.setdefault(a, []).append(b)
        neighbours.setdefault(b, []).append(a)

    corners = [node for node, nbrs in neighbours.items() if len(nbrs) == 3]
    if len(corners) != 8:
        raise ValueError(f"Expected 8 cube corners, found {len(corners)}.")

    def _faces(a, b):
        return edge_faces[tuple(sorted((a, b)))]

    lines = []
    seen = set()
    for start in corners:
        for nxt in neighbours[start]:
            path = [start, nxt]
            # Walk straight on: the next edge shares no face with the previous one
            while len(neighbours[path[-1]]) != 3:
                prev, cur = path[-2], path[-1]
                (nxt,) = [
                    n
                    for n in neighbours[cur]
                    if n != prev and not (_faces(prev, cur) & _faces(cur, n))
                ]
                path.append(nxt)
            key = tuple(sorted((path[0], path[-1]))) + (path[1],)
            key_rev = tuple(sorted((path[0], path[-1]))) + (path[-2],)
            if key in seen or key_rev in seen:
                continue
            seen.add(key)
            lons = np.unwrap(node_lons[path], period=360)
            lines.append(np.column_stack([lons, node_lats[path]]))
    if len(lines) != 12:
        raise ValueError(f"Expected 12 panel edges, found {len(lines)}.")
    return lines


def plot_cube_panel_edges(
    obj, ax=None, transform="auto", color="k", linewidth=1, **kwargs
):
    """
    Plot the edges of the six panels of a cubed-sphere mesh as lines.

    Parameters
    ----------
    obj: iris.mesh.MeshXY or iris.cube.Cube
        Mesh or cube, see `get_face_nodes()`.
    ax: matplotlib.axes.Axes, optional
        Axes to plot on (e.g. cartopy GeoAxes). Defaults to the current axes.
    transform: cartopy.crs.CRS, "auto" or None, optional
        Coordinate system of the lon-lat lines.
        - "auto" (default): `ccrs.Geodetic()` (great circle arcs between mesh
          nodes) if `ax` is a cartopy GeoAxes, otherwise None.
        - None: plot on regular (non-cartopy) matplotlib axes, with longitude
          and latitude in degrees as x and y.
        - a cartopy CRS, e.g. `ccrs.PlateCarree()` for straight lines in lon-lat.
    color: str, optional
        Line colour.
    linewidth: float, optional
        Line width.
    **kwargs:
        Passed to `matplotlib.collections.LineCollection`.

    Returns
    -------
    matplotlib.collections.LineCollection
    """
    if ax is None:
        ax = plt.gca()
    if isinstance(transform, str) and transform == "auto":
        transform = ccrs.Geodetic() if hasattr(ax, "projection") else None
    lines = cube_panel_edges(obj)
    if transform is None:
        # Plain axes: duplicate lines crossing the dateline, shifted by 360 degrees
        # (cartopy wraps them automatically, so this is not needed for GeoAxes)
        lines = (
            lines
            + [ln - [360, 0] for ln in lines if ln[:, 0].max() > 180]
            + [ln + [360, 0] for ln in lines if ln[:, 0].min() < -180]
        )
    else:
        kwargs["transform"] = transform
    coll = LineCollection(lines, colors=color, linewidths=linewidth, **kwargs)
    ax.add_collection(coll)
    if transform is None:
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
    return coll

## Maps for a group of simulations on the native mesh

Time-mean longitude-latitude maps on a pressure level, one panel per experiment of a group in `common.GROUPS`
with a shared colour scale, following the maps in `Group-Cross-Sections.ipynb`.
Each cell of the cubed-sphere mesh (`lfric_diag_main*.nc`) is drawn as a polygon.

In [ ]:
GROUP_KEY = "dhj_all"
# Subset of the group's simulations to plot (all of them if None)
CASES = None
N_DAYS_MEAN = 1  # length of the averaging period
NCOLS = 5  # number of panels per row
LEVEL = 1e6  # Pa
SHOW_PANEL_EDGES = True  # overlay the edges of the six cubed-sphere panels
MAP_PROJ = ccrs.PlateCarree(central_longitude=0)
SAVE_FIGS = True

EXPS = list(GROUPS[GROUP_KEY].simulations) if CASES is None else CASES
# Average all experiments over the same days, at the end of the shortest run
END_DAY = min(EXPERIMENTS[exp_key].run_length for exp_key in EXPS)
START_DAY = END_DAY - N_DAYS_MEAN
if N_DAYS_MEAN > 1:
    PERIOD_STR = f"days {START_DAY}-{END_DAY}"
else:
    PERIOD_STR = f"day {END_DAY}"
print(*EXPS, sep="\n")
print()
print(PERIOD_STR)

In [ ]:
ds_all = {}
for exp_key in EXPS:
    ds_all[exp_key] = time_mean(
        load_exp_days(exp_key, START_DAY, END_DAY, file_glob="lfric_diag_main*.nc")
    )
    print(exp_key)

In [ ]:
DIAGS = {
    "temp": Diag(
        title="Air Temperature",
        units="$K$",
        recipe=lambda cl: cl.extract_cube("air_temperature"),
        kw_plt={"cmap": "plasma"},
    ),
    "u": Diag(
        title="Eastward Wind",
        units="$km$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("eastward_wind"),
        cnorm=True,
        kw_plt={"cmap": "RdBu_r"},
    ),
    "w": Diag(
        title="Upward Wind",
        units="$m$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("upward_air_velocity"),
        cnorm=True,
        kw_plt={"cmap": "PuOr_r"},
    ),
    "vort": Diag(
        title="Vertical Vorticity",
        units="$10^{-5}$ $s^{-1}$",
        recipe=lambda cl: cl.extract_cube("vertical_vorticity"),
        cnorm=True,
        kw_plt={"cmap": "seismic"},
    ),
}


def exp_title(exp_key):
    """Experiment title, or its label if the titles in `EXPS` are not unique."""
    titles = [EXPERIMENTS[k].title for k in EXPS]
    if len(set(titles)) < len(titles):
        return EXPERIMENTS[exp_key].label
    return EXPERIMENTS[exp_key].title


def plot_group_mesh_maps(
    diag_key,
    level=LEVEL,
    show_panel_edges=SHOW_PANEL_EDGES,
    ncol=NCOLS,
    add_cbar=True,
):
    """Plot a map on a pressure level on the native mesh for each experiment in `EXPS`."""
    diag = DIAGS[diag_key]
    cubes = []
    for exp_key in EXPS:
        cube = interp_to_plev(
            diag.recipe(ds_all[exp_key]),
            ds_all[exp_key].extract_cube("air_pressure"),
            level,
        )
        cube.convert_units(tex2cf_units(diag.units))
        cubes.append(cube)
    kw_plt, levels = common_kw_plt(diag, cubes)

    ncols = min(ncol, len(EXPS))
    nrows = math.ceil(len(EXPS) / ncols)
    # Maps in rows of `ncols` (empty cells are "."), and a colour bar spanning all rows
    mosaic = [list(EXPS[i : i + ncols]) for i in range(0, len(EXPS), ncols)]
    mosaic[-1] += ["."] * (ncols - len(mosaic[-1]))
    if add_cbar:
        mosaic = [row + ["cbar"] for row in mosaic]
    # The "compressed" layout removes the spare space around the fixed-aspect maps,
    # so that the title is close to them
    fig = plt.figure(figsize=(2.5 * ncols, 1.5 * nrows + 0.5), layout="compressed")
    axd = fig.subplot_mosaic(
        mosaic,
        width_ratios=[1] * ncols + ([0.03] if add_cbar else []),
        per_subplot_kw={tuple(EXPS): {"projection": MAP_PROJ}},
    )

    iletters = subplot_label_generator()
    for i, (exp_key, cube) in enumerate(zip(EXPS, cubes)):
        ax = axd[exp_key]
        ax.set_title(f"({next(iletters)})", loc="left", fontsize="medium")
        ax.set_title(
            exp_title(exp_key),
            fontsize="medium",
            color=EXPERIMENTS[exp_key].kw_plt["color"],
        )
        ax.set_global()
        im = plot_mesh_polygons(cube, ax=ax, antialiased=False, **kw_plt)
        if show_panel_edges:
            plot_cube_panel_edges(cube, ax=ax, linewidth=0.5)
        ax.gridlines(
            draw_labels=["bottom", "left"] if i % ncols == 0 else ["bottom"],
            xlocs=np.arange(-180, 181, 60),
            ylocs=np.arange(-90, 91, 30),
            linestyle="--",
            alpha=0.5,
            xlabel_style={"size": "x-small"},
            ylabel_style={"size": "x-small"},
        )
    if add_cbar:
        cbar = fig.colorbar(
            im, cax=axd["cbar"], orientation="vertical", extend="both", ticks=levels
        )

    fig.suptitle(
        GROUPS[GROUP_KEY].title
        + "\n"
        + " | ".join([
            f"{diag.title} [{diag.units}]",
            f"Level: {unit_format(level * PA_TO_BAR, 'bar')}",
            PERIOD_STR,
        ]),
        fontsize="medium",
    )
    if add_cbar:
        # Freeze the layout and make the colour bar as tall as the maps
        align_cbar_with_axes(cbar, [axd[exp_key] for exp_key in EXPS])

    if SAVE_FIGS:
        figsave(
            fig,
            paths.drafts / "xsec" / f"{GROUP_KEY}__{diag_key}__cs__{level:.0f}pa"
            f"__days_{START_DAY}_{END_DAY}",
        )
    return fig

In [ ]:
for diag_key in ["u", "temp", "vort", "w"]:
    plot_group_mesh_maps(diag_key)

## Testing

In [ ]:
# group_key = "dhj_all"
# ds_rg = {}
# for exp_key in tqdm(GROUPS[group_key].simulations):
#     ds_rg[exp_key] =
exp_key = "dhj_c48_l66_s0p5_p90"

ds_init = load_lfric_raw(
    paths.data_work
    / "lfric"
    / EXPERIMENTS[exp_key].group
    / EXPERIMENTS[exp_key].label
    / "lfric_initial.nc",
    # callback=partial(
    #     lfric_callback_uniform_height,
    #     model_top_height=EXPERIMENTS[exp_label].const.domain_height.data,
    # ),
)
print(ds_init)

In [ ]:
cube = ds_init.extract_cube("grid_cell_area")

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=0))
ax.set_global()
ax.gridlines(draw_labels=True)
# mappable = plot_mesh_polygons(cube, ax=ax, norm=mcol.CenteredNorm(), cmap="seismic")
map_scatter(
    ds_init.extract_cube("grid_cell_area"),
    s=1,
    ax=ax,
    transform=ccrs.PlateCarree(),
)
plot_cube_panel_edges(cube, ax=ax)
# fig.colorbar(mappable, ax=ax, shrink=0.7, label=f"{cube.name()} [{cube.units}]")
ax.set_title(cube.name())

In [ ]:
exp_key = "dhj_c48_l66_s0p5_p90"
ds_main = load_lfric_raw(
    sorted(
        (
            paths.data_work
            / "lfric"
            / EXPERIMENTS[exp_key].group
            / EXPERIMENTS[exp_key].label
        ).glob("lfric_diag_latlon*.nc")  # "lfric_diag_main*.nc"
    )[-2:],
    callback=partial(
        lfric_callback_uniform_height,
        model_top_height=EXPERIMENTS[exp_key].const.domain_height.data,
    ),
)
print(ds_main)

In [ ]:
cube = ds_main.extract_cube("upward_air_velocity")[-1, 20, ...]
hcross(roll_cube_pm180(cube, model=lfric), norm=mcol.CenteredNorm(), cmap="seismic")

In [ ]:
cube = ds_main.extract_cube("upward_air_velocity")[-1, ...]

plt.pcolormesh(cube.data[:, 45, ...], norm=mcol.CenteredNorm(), cmap="seismic")
plt.colorbar()

In [ ]:
cube = ds_main.extract_cube("vertical_vorticity")[-1, -25, ...]

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.EqualEarth(central_longitude=0))
ax.set_global()
ax.gridlines(draw_labels=True)
mappable = plot_mesh_polygons(cube, ax=ax, norm=mcol.CenteredNorm(), cmap="seismic")
plot_cube_panel_edges(cube, ax=ax)
fig.colorbar(mappable, ax=ax, shrink=0.7, label=f"{cube.name()} [{cube.units}]")
ax.set_title(cube.name())
mappable.get_clim()

In [ ]:
cube = ds_main.extract_cube("vertical_vorticity")[-1, 37, ...]

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.EqualEarth(central_longitude=0))
ax.set_global()
ax.gridlines(draw_labels=True)
# Roll the lat-lon data to -180...180, so the gap in the contours is at the map edge
cube_pm180 = roll_cube_pm180(cube, model=lfric)
mappable = ax.contourf(
    cube_pm180.coord("longitude").points,
    cube_pm180.coord("latitude").points,
    cube_pm180.data,
    norm=mcol.CenteredNorm(),
    cmap="seismic",
    **CART_TRANS,
)
# plot_cube_panel_edges(cube, ax=ax)
fig.colorbar(mappable, ax=ax, shrink=0.7, label=f"{cube.name()} [{cube.units}]")
ax.set_title(cube.name())
mappable.get_clim()

In [ ]:
m = plot_mesh_polygons(cube, norm=mcol.CenteredNorm(), cmap="seismic")
cbar = plt.colorbar(m)

In [ ]:
map_scatter(cube, norm=mcol.CenteredNorm(), cmap="seismic")